In [1]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm  import tqdm
import json
import ast
import numpy as np
import gemmi
from mlindex.dataset_generation.EntryHelpers import spacegroup_to_symmetry 
from mlindex.utilities.Reindexing import reindex_entry_monoclinic
from mlindex.utilities.Reindexing import reindex_entry_orthorhombic
from mlindex.utilities.Reindexing import reindex_entry_triclinic
from mlindex.utilities.Reindexing import hexagonal_to_rhombohedral_unit_cell

from parse_opxrd import pick_peaks
from parse_opxrd import get_symmetry

from cctbx import crystal
from cctbx import uctbx
from cctbx.sgtbx.lattice_symmetry import metric_subgroups

In [2]:
root = '/global/cfs/cdirs/m4064/dwmoreau/opxrd'

In [3]:
def get_q(info):
    unit_cell = info['unit_cell'].copy()
    unit_cell[3:] *= 180/np.pi
    input_symmetry = crystal.symmetry(
        unit_cell=uctbx.unit_cell(parameters=list(unit_cell)),
        space_group_symbol=info['spacegroup'] if info['spacegroup'] else 'P 1'
    )
    # Do it
    groups = metric_subgroups(
        input_symmetry,
        0.1,
        enforce_max_delta_for_generated_two_folds=True
    )
    c = groups.result_groups[0]['best_subsym']
    miller_indices = c.build_miller_set(d_min=1, d_max=40, anomalous_flag=False)
    return 1/np.array(list(miller_indices.d_spacings().data()))


In [7]:
def parse_file(file_name):
    df = pd.read_json(file_name)
    data = json.loads(df['label'][0])
    phases = data['phases']
    if len(phases) > 0:
        phase = json.loads(data['phases'][0])
        spacegroup_number = phase['spacegroup']
        if spacegroup_number:
            spacegroup = gemmi.SpaceGroup(spacegroup_number).hm
        else:
            spacegroup = None
        unit_cell_entry = np.array(ast.literal_eval(phase['lattice']))
        unit_cell_entry[3:] *= np.pi/180
    else:
        spacegroup_number = None
        spacegroup = None
        unit_cell_entry = None
        unit_cell = None
    xray_info = json.loads(data.get("xray_info") or "{}")

    if spacegroup is None:
        unit_cell, crystal_system = get_symmetry(unit_cell_entry)
        crystal_system = crystal_system.lower()
        lattice_system = crystal_system
        if crystal_system == 'triclinic':
            bravais_lattice = 'aP'
        elif crystal_system == 'monoclinic':
            bravais_lattice = 'mP'
        elif crystal_system == 'orthorhombic':
            bravais_lattice = 'oP'
        elif crystal_system == 'tetragonal':
            bravais_lattice = 'tP'
        elif crystal_system == 'cubic':
            bravais_lattice = 'cP'
        elif crystal_system == 'hexagonal':
            bravais_lattice = 'hP'
        elif crystal_system == 'rhombohedral':
            bravais_lattice = 'hR'
        else:
            print(crystal_system)
            assert False
    else:
        bravais_lattice, crystal_family, crystal_system, lattice_system = spacegroup_to_symmetry(int(spacegroup_number))
        if lattice_system == 'triclinic':
            bravais_lattice = 'aP'
            unit_cell, _ = reindex_entry_triclinic(unit_cell_entry, space='direct')
        elif lattice_system == 'monoclinic':
            unit_cell = np.array(unit_cell_entry)
            unit_cell, spacegroup, _ = reindex_entry_monoclinic(
                np.array(unit_cell),
                spacegroup,
                space='direct'
            )
        elif lattice_system == 'orthorhombic':
            spacegroup, _, unit_cell, _ = reindex_entry_orthorhombic(
                np.array(unit_cell_entry),
                spacegroup,
                int(spacegroup_number)
                )
        elif lattice_system == 'rhombohedral':
            if np.all(unit_cell_entry[3:] == [np.pi/2, np.pi/2, 2*np.pi/3]):
                unit_cell, _ = hexagonal_to_rhombohedral_unit_cell(unit_cell_entry)
            else:
                unit_cell = unit_cell_entry
        else:
            unit_cell = unit_cell_entry
    info =  {
        "file_name": file_name,
        "primary_wavelength": float(xray_info["primary_wavelength"]) if xray_info.get("primary_wavelength") else None,
        "secondary_wavelength": float(xray_info["secondary_wavelength"]) if xray_info.get("secondary_wavelength") else None,
        "spacegroup": spacegroup,
        "unit_cell": unit_cell,
        "unit_cell_entry": unit_cell_entry,
        "theta2": df['two_theta_values'],
        "I": df['intensities'],
        "q": None,
    }
    valid = info["I"] > 0
    info["I"] = np.array(info["I"][valid])
    info["theta2"] = np.array(info["theta2"][valid])
    if info['primary_wavelength']:
        info['q'] = 2 * np.sin(np.pi/180 * info['theta2']/2) / info['primary_wavelength']

    info['bravais_lattice'] = bravais_lattice
    info['lattice_system'] = lattice_system
    return info

def plot(info, q_found=None, q_lattice=None):
    fig, axes = plt.subplots(1, 1, figsize=(15, 4))
    axes.plot(info['q'], info['I'])
    ylims = axes.get_ylim()
    if not q_found is None:
        for q in q_found:
            axes.plot([q, q], ylims, color=[0, 0, 0])
    if not q_lattice is None:
        for q in q_lattice:
            axes.plot([q, q], [ylims[0], 0.5*ylims[1]], color=[1, 0, 0])
    axes.set_ylim(ylims)
    axes.set_xlim([axes.get_xlim()[0], q_found[-1]])
    plt.show()

source = 'CNRS'
file_names = sorted(os.listdir(root + '/' + source))
output_data = []
all_lattices = []
for file_name in tqdm(file_names):
    if file_name.endswith('.json'):
        info = parse_file(root + '/' + source + '/' + file_name)
        #print(root + '/' + source + '/' + file_name)
        #print(info['primary_wavelength'], info['secondary_wavelength'])
        #print(info['unit_cell'], info['spacegroup'])
        #print()
        all_lattices.append(info['unit_cell_entry'])
        if not info['q'] is None:
            peaks, bg, diag = pick_peaks(
                info['q'], info['I'],
                snr_threshold=6,
                min_relative_height=0.015,
                profile="pseudo_voigt",
                bg_width=None,
                bg_iterations=20,
                smooth_window=10,  
                merge_distance_factor=4,
                show_diagnostics=False,
            )

            try:
                q_lattice = get_q(info)
                q_found = np.array([p['center'] for p in peaks])
                sigma_found = np.array([p['sigma'] for p in peaks])
            except:
                q_found = [0]
                sigma_found = [0]
            if len(q_found) >= 20:
                info['peak_positions'] = q_found
                info['peak_sigmas'] = sigma_found
                info['file_name'] = root + '/' + source + '/' + file_name
                output_data.append(info)
            #plot(info, q_found, q_lattice)  
            #plot(info)
np.save('CNRS_lattices.npy', np.array(all_lattices))
output_data = pd.DataFrame(output_data)
output_data.to_json(f'{source}_output_data.json')

/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/data/opxrd/parse_opxrd.py:211: RuntimeWarning: divide by zero encountered in divide
  g = np.exp(-0.5 * ((x - center) / sigma) ** 2)
/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/data/opxrd/parse_opxrd.py:211: RuntimeWarning: invalid value encountered in divide
  g = np.exp(-0.5 * ((x - center) / sigma) ** 2)
/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/data/opxrd/parse_opxrd.py:212: RuntimeWarning: divide by zero encountered in divide
  l = 1.0 / (1.0 + ((x - center) / sigma) ** 2)
/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/data/opxrd/parse_opxrd.py:212: RuntimeWarning: invalid value encountered in divide
  l = 1.0 / (1.0 + ((x - center) / sigma) ** 2)
100%|██████████| 1052/1052 [26:46<00:00,  1.53s/it] 
